# API-Football Datenakquise — FC Thun / Super League

Dieses Notebook lädt Rohdaten von der API-Football v3 (via RapidAPI) für die Schweizer Super League
und speichert sie als CSV-Dateien in `data_acquisition/raw/`.

**Endpoints:**
- `/standings` → Tabelle (Rangliste, Punkte, Tore, Form)
- `/fixtures` → Alle Spielresultate der Saison
- `/teams` → Team-IDs und Metadaten
- `/teams/statistics` → Teamstatistiken pro Verein (Tore, Schüsse, Karten, etc.)

**Voraussetzung:** `.env`-Datei im Projekt-Root mit:
```
API_FOOTBALL_KEY="dein_key"
API_FOOTBALL_URL="https://api-football-v1.p.rapidapi.com/v3"
```

**Rate Limit:** 100 Requests/Tag (kostenloser Plan). Dieses Notebook benötigt ca. 12–14 Requests.

## 1. Setup & Imports

In [1]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Pfad zur .env-Datei (Projekt-Root ist zwei Ebenen über diesem Notebook)
env_path = Path("__file__").resolve().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)

API_KEY = os.environ["API_FOOTBALL_KEY"]
BASE_URL = os.environ["API_FOOTBALL_URL"]

HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": "api-football-v1.p.rapidapi.com",
}

# Schweizer Super League
LEAGUE_ID = 207
SEASON = 2025

RAW_DIR = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

print(f"Base URL: {BASE_URL}")
print(f"API Key geladen: {'Ja' if API_KEY else 'NEIN – .env prüfen!'}")
print(f"Ausgabeverzeichnis: {RAW_DIR.resolve()}")

ModuleNotFoundError: No module named 'dotenv'

## 2. Hilfsfunktion für API-Requests

In [ ]:
def api_get(endpoint: str, params: dict) -> dict:
    """Führt einen GET-Request an die API-Football API durch.
    
    Gibt das 'response'-Feld des JSON zurück.
    Wirft einen Fehler bei HTTP-Fehler oder leerem Response.
    """
    url = f"{BASE_URL}/{endpoint}"
    resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    data = resp.json()

    if data.get("errors"):
        raise ValueError(f"API-Fehler bei /{endpoint}: {data['errors']}")

    remaining = resp.headers.get("x-ratelimit-requests-remaining", "?")
    print(f"  ✓ /{endpoint} | Verbleibende Requests heute: {remaining}")
    return data["response"]

## 3. Standings (Tabelle)

In [ ]:
print("Lade Standings...")
standings_raw = api_get("standings", {"league": LEAGUE_ID, "season": SEASON})

# Struktur: response[0].league.standings[0] → Liste der Teams
standing_list = standings_raw[0]["league"]["standings"][0]

rows = []
for entry in standing_list:
    rows.append({
        "rank":            entry["rank"],
        "team_id":         entry["team"]["id"],
        "team_name":       entry["team"]["name"],
        "points":          entry["points"],
        "goals_diff":      entry["goalsDiff"],
        "form":            entry["form"],
        "status":          entry["status"],
        "description":     entry.get("description", ""),
        # Gesamt
        "played":          entry["all"]["played"],
        "win":             entry["all"]["win"],
        "draw":            entry["all"]["draw"],
        "lose":            entry["all"]["lose"],
        "goals_for":       entry["all"]["goals"]["for"],
        "goals_against":   entry["all"]["goals"]["against"],
        # Heimspiele
        "home_played":     entry["home"]["played"],
        "home_win":        entry["home"]["win"],
        "home_draw":       entry["home"]["draw"],
        "home_lose":       entry["home"]["lose"],
        "home_goals_for":  entry["home"]["goals"]["for"],
        "home_goals_against": entry["home"]["goals"]["against"],
        # Auswärtsspiele
        "away_played":     entry["away"]["played"],
        "away_win":        entry["away"]["win"],
        "away_draw":       entry["away"]["draw"],
        "away_lose":       entry["away"]["lose"],
        "away_goals_for":  entry["away"]["goals"]["for"],
        "away_goals_against": entry["away"]["goals"]["against"],
        "update":          entry["update"],
    })

df_standings = pd.DataFrame(rows)
df_standings.to_csv(RAW_DIR / "standings.csv", index=False)

print(f"\nStandings gespeichert: {len(df_standings)} Teams")
df_standings[["rank", "team_name", "points", "played", "win", "draw", "lose", "goals_for", "goals_against"]]

## 4. Teams (IDs für spätere Requests)

In [ ]:
print("Lade Teams...")
teams_raw = api_get("teams", {"league": LEAGUE_ID, "season": SEASON})

rows = []
for entry in teams_raw:
    t = entry["team"]
    v = entry["venue"]
    rows.append({
        "team_id":      t["id"],
        "team_name":    t["name"],
        "team_code":    t.get("code", ""),
        "country":      t.get("country", ""),
        "founded":      t.get("founded", ""),
        "national":     t.get("national", False),
        "logo":         t.get("logo", ""),
        "venue_name":   v.get("name", ""),
        "venue_city":   v.get("city", ""),
        "venue_capacity": v.get("capacity", ""),
    })

df_teams = pd.DataFrame(rows)
df_teams.to_csv(RAW_DIR / "teams.csv", index=False)

TEAM_IDS = df_teams["team_id"].tolist()
print(f"\nTeams gespeichert: {len(df_teams)} Teams")
print(f"Team-IDs: {TEAM_IDS}")
df_teams[["team_id", "team_name", "venue_name", "venue_city", "venue_capacity"]]

## 5. Fixtures (Spielresultate)

In [ ]:
print("Lade Fixtures...")
fixtures_raw = api_get("fixtures", {"league": LEAGUE_ID, "season": SEASON})

rows = []
for entry in fixtures_raw:
    f  = entry["fixture"]
    lg = entry["league"]
    ht = entry["teams"]["home"]
    at = entry["teams"]["away"]
    g  = entry["goals"]
    sc = entry["score"]
    rows.append({
        "fixture_id":       f["id"],
        "date":             f["date"],
        "referee":          f.get("referee", ""),
        "venue_name":       f["venue"].get("name", ""),
        "venue_city":       f["venue"].get("city", ""),
        "status_long":      f["status"]["long"],
        "status_short":     f["status"]["short"],
        "round":            lg["round"],
        "home_team_id":     ht["id"],
        "home_team_name":   ht["name"],
        "home_team_winner": ht["winner"],
        "away_team_id":     at["id"],
        "away_team_name":   at["name"],
        "away_team_winner": at["winner"],
        "goals_home":       g["home"],
        "goals_away":       g["away"],
        "score_ht_home":    sc["halftime"]["home"],
        "score_ht_away":    sc["halftime"]["away"],
        "score_ft_home":    sc["fulltime"]["home"],
        "score_ft_away":    sc["fulltime"]["away"],
    })

df_fixtures = pd.DataFrame(rows)
df_fixtures.to_csv(RAW_DIR / "fixtures.csv", index=False)

played = df_fixtures[df_fixtures["status_short"] == "FT"]
print(f"\nFixtures gespeichert: {len(df_fixtures)} Spiele total, {len(played)} abgeschlossen")
df_fixtures.head(5)

## 6. Team Statistics (pro Verein)

Dieser Block sendet **einen Request pro Team** (~10 Requests).  
Bei 100 Req/Tag Budget auf den täglichen Stand achten!

In [ ]:
def flatten_team_stats(resp: dict, team_id: int, team_name: str) -> dict:
    """Flacht das verschachtelte Team-Statistics-JSON in eine flache Zeile."""
    g = resp.get("goals", {})
    f = resp.get("fixtures", {})
    b = resp.get("biggest", {})
    cs = resp.get("clean_sheet", {})
    fts = resp.get("failed_to_score", {})
    pen = resp.get("penalty", {})

    row = {
        "team_id":                team_id,
        "team_name":              team_name,
        # Spiele
        "fixtures_played_home":   f.get("played", {}).get("home", None),
        "fixtures_played_away":   f.get("played", {}).get("away", None),
        "fixtures_played_total":  f.get("played", {}).get("total", None),
        "fixtures_wins_home":     f.get("wins", {}).get("home", None),
        "fixtures_wins_away":     f.get("wins", {}).get("away", None),
        "fixtures_wins_total":    f.get("wins", {}).get("total", None),
        "fixtures_draws_home":    f.get("draws", {}).get("home", None),
        "fixtures_draws_away":    f.get("draws", {}).get("away", None),
        "fixtures_draws_total":   f.get("draws", {}).get("total", None),
        "fixtures_loses_home":    f.get("loses", {}).get("home", None),
        "fixtures_loses_away":    f.get("loses", {}).get("away", None),
        "fixtures_loses_total":   f.get("loses", {}).get("total", None),
        # Tore geschossen
        "goals_for_home":         g.get("for", {}).get("total", {}).get("home", None),
        "goals_for_away":         g.get("for", {}).get("total", {}).get("away", None),
        "goals_for_total":        g.get("for", {}).get("total", {}).get("total", None),
        "goals_for_avg_home":     g.get("for", {}).get("average", {}).get("home", None),
        "goals_for_avg_away":     g.get("for", {}).get("average", {}).get("away", None),
        "goals_for_avg_total":    g.get("for", {}).get("average", {}).get("total", None),
        # Tore kassiert
        "goals_against_home":     g.get("against", {}).get("total", {}).get("home", None),
        "goals_against_away":     g.get("against", {}).get("total", {}).get("away", None),
        "goals_against_total":    g.get("against", {}).get("total", {}).get("total", None),
        "goals_against_avg_home": g.get("against", {}).get("average", {}).get("home", None),
        "goals_against_avg_away": g.get("against", {}).get("average", {}).get("away", None),
        "goals_against_avg_total": g.get("against", {}).get("average", {}).get("total", None),
        # Grösste Siege/Niederlagen
        "biggest_wins_home":      b.get("wins", {}).get("home", ""),
        "biggest_wins_away":      b.get("wins", {}).get("away", ""),
        "biggest_loses_home":     b.get("loses", {}).get("home", ""),
        "biggest_loses_away":     b.get("loses", {}).get("away", ""),
        "biggest_streak_wins":    b.get("streak", {}).get("wins", None),
        "biggest_streak_draws":   b.get("streak", {}).get("draws", None),
        "biggest_streak_loses":   b.get("streak", {}).get("loses", None),
        "biggest_goals_for_home": b.get("goals", {}).get("for", {}).get("home", None),
        "biggest_goals_for_away": b.get("goals", {}).get("for", {}).get("away", None),
        "biggest_goals_against_home": b.get("goals", {}).get("against", {}).get("home", None),
        "biggest_goals_against_away": b.get("goals", {}).get("against", {}).get("away", None),
        # Clean sheets & Torlos
        "clean_sheet_home":       cs.get("home", None),
        "clean_sheet_away":       cs.get("away", None),
        "clean_sheet_total":      cs.get("total", None),
        "failed_to_score_home":   fts.get("home", None),
        "failed_to_score_away":   fts.get("away", None),
        "failed_to_score_total":  fts.get("total", None),
        # Penalties
        "penalty_scored_total":   pen.get("scored", {}).get("total", None),
        "penalty_missed_total":   pen.get("missed", {}).get("total", None),
    }
    return row


print(f"Lade Team-Statistiken für {len(TEAM_IDS)} Teams...\n")
team_stat_rows = []

for team_id in TEAM_IDS:
    team_name = df_teams.loc[df_teams["team_id"] == team_id, "team_name"].values[0]
    print(f"  → {team_name} (ID {team_id})")
    resp = api_get("teams/statistics", {
        "league": LEAGUE_ID,
        "season": SEASON,
        "team":   team_id,
    })
    row = flatten_team_stats(resp, team_id, team_name)
    team_stat_rows.append(row)
    time.sleep(0.5)  # kleiner Puffer zwischen Requests

df_team_stats = pd.DataFrame(team_stat_rows)
df_team_stats.to_csv(RAW_DIR / "team_statistics.csv", index=False)

print(f"\nTeam-Statistiken gespeichert: {len(df_team_stats)} Zeilen × {len(df_team_stats.columns)} Spalten")
df_team_stats[["team_name", "fixtures_played_total", "fixtures_wins_total",
               "goals_for_total", "goals_against_total", "clean_sheet_total"]]

## 7. Übersicht gespeicherter Dateien

In [ ]:
print("Gespeicherte Rohdaten in data_acquisition/raw/:\n")
for csv_file in sorted(RAW_DIR.glob("*.csv")):
    df = pd.read_csv(csv_file)
    size_kb = csv_file.stat().st_size / 1024
    print(f"  {csv_file.name:<30} {len(df):>4} Zeilen × {len(df.columns):>2} Spalten   ({size_kb:.1f} KB)")

print("\nDatenakquise abgeschlossen.")